In [1]:
import pandas as pd

# Load the user's specific Amazon dataset
try:
    df = pd.read_csv("amazon.csv")
    print("Dataset loaded successfully.")
    
    print("\n--- 1. DATASET DIMENSIONS ---")
    print(f"Total Records (Rows): {df.shape[0]}")
    print(f"Total Attributes (Columns): {df.shape[1]}")
    
    print("\n--- 2. DETECTED COLUMN NAMES ---")
    print(list(df.columns))
    
    print("\n--- 3. DATA TYPES & MISSING VALUES ---")
    print(df.isna().sum())
    
    print("\n--- 4. FIRST 3 ROWS PREVIEW ---")
    print(df.head(3))

except FileNotFoundError:
    print("Error: Could not find 'amazon_sales.csv' in this folder. Please verify the filename.")


Dataset loaded successfully.

--- 1. DATASET DIMENSIONS ---
Total Records (Rows): 1465
Total Attributes (Columns): 16

--- 2. DETECTED COLUMN NAMES ---
['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'img_link', 'product_link']

--- 3. DATA TYPES & MISSING VALUES ---
product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 0
rating_count           2
about_product          0
user_id                0
user_name              0
review_id              0
review_title           0
review_content         0
img_link               0
product_link           0
dtype: int64

--- 4. FIRST 3 ROWS PREVIEW ---
   product_id                                       product_name  \
0  B07JW9H4J1  Wayona Nylon Braided USB to Lightni

In [2]:
import pandas as pd
import numpy as np

# 1. Load data safely
df = pd.read_csv("amazon.csv")

# 2. Clean pricing columns by stripping currency signs and commas
for col in ['discounted_price', 'actual_price']:
    # Strip common currency indicators and commas, then convert to numeric
    df[col] = df[col].astype(str).str.replace(r'[^\d.]', '', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Clean discount percentage column
df['discount_percentage'] = df['discount_percentage'].astype(str).str.replace('%', '')
df['discount_percentage'] = pd.to_numeric(df['discount_percentage'], errors='coerce') / 100

# 4. Clean rating column (handles cases where a rating might be a text placeholder)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# 5. Handle missing values in rating_count and clean formatting commas
df['rating_count'] = df['rating_count'].astype(str).str.replace(',', '')
df['rating_count'] = pd.to_numeric(df['rating_count'], errors='coerce')
df['rating_count'] = df['rating_count'].fillna(0).astype(int)

# 6. Engineer high-utility e-commerce metrics for the dashboard
# Estimate total popularity engagement and calculate price drop margins
df['revenue_proxy'] = df['discounted_price'] * df['rating_count']
df['discount_value'] = df['actual_price'] - df['discounted_price']

# Extract the main primary category from the hierarchical category text path
df['root_category'] = df['category'].astype(str).apply(lambda x: x.split('|')[0] if '|' in x else x)

print("Data Cleaning Pipeline executed successfully.")
print(f"Cleaned Numeric Range: Prices range from {df['discounted_price'].min()} to {df['discounted_price'].max()}")
print(f"Primary Root Categories Identified: {df['root_category'].unique()[:5]}")


Data Cleaning Pipeline executed successfully.
Cleaned Numeric Range: Prices range from 39.0 to 77990.0
Primary Root Categories Identified: ['Computers&Accessories' 'Electronics' 'MusicalInstruments'
 'OfficeProducts' 'Home&Kitchen']


In [3]:
# Extract only the first element from the split path list
df['root_category'] = df['category'].astype(str).apply(lambda x: x.split('|')[0] if '|' in x else x)
print("Root categories successfully converted to text elements.")


Root categories successfully converted to text elements.


In [4]:
import plotly.express as px

print("--- 1. MARKET OVERVIEW BY ROOT CATEGORY ---")
# Group by category to analyze market penetration
category_analysis = df.groupby('root_category').agg(
    total_products=('product_id', 'count'),
    avg_discounted_price=('discounted_price', 'mean'),
    avg_discount_pct=('discount_percentage', 'mean'),
    avg_customer_rating=('rating', 'mean'),
    total_reviews=('rating_count', 'sum')
).sort_values(by='total_products', ascending=False).reset_index()

display(category_analysis.round(2))

print("\n--- 2. GENERATING INSIGHT VISUALIZATIONS ---")

# Visualization A: Market Share distribution based on item counts
fig_share = px.bar(
    category_analysis, 
    x='total_products', 
    y='root_category', 
    orientation='h',
    title='Product Volume Share by Root Category',
    labels={'total_products': 'Number of Products Listed', 'root_category': 'Category'},
    template='plotly_white'
)
fig_share.update_layout(yaxis={'categoryorder':'total ascending'})
fig_share.show()

# Visualization B: Pricing Elasticity Scatter Plot
fig_scatter = px.scatter(
    df, 
    x='discount_percentage', 
    y='rating', 
    size='rating_count', 
    color='root_category',
    title='Rating and Engagement vs. Applied Discount Percentage',
    labels={'discount_percentage': 'Discount Applied (%)', 'rating': 'Customer Score (1-5)'},
    template='plotly_white',
    hover_name='product_name'
)
fig_scatter.show()


--- 1. MARKET OVERVIEW BY ROOT CATEGORY ---


,root_category,total_products,avg_discounted_price,avg_discount_pct,avg_customer_rating,total_reviews
0,Electronics,526,5965.89,0.51,4.08,15778848
1,Computers&Accessories,453,842.65,0.54,4.15,7728689
2,Home&Kitchen,448,2330.62,0.40,4.04,2991069
3,OfficeProducts,31,301.58,0.12,4.31,149675
4,HomeImprovement,2,337.00,0.57,4.25,8566
5,MusicalInstruments,2,638.00,0.46,3.90,88882
6,Car&Motorbike,1,2339.00,0.42,3.80,1118
7,Health&PersonalCare,1,899.00,0.53,4.00,3663
8,Toys&Games,1,150.00,0.00,4.30,15867



--- 2. GENERATING INSIGHT VISUALIZATIONS ---


In [5]:
df.to_csv("amazon_cleaned.csv", index=False)
print("Export complete: Use 'amazon_cleaned.csv' inside your dashboard application.")


Export complete: Use 'amazon_cleaned.csv' inside your dashboard application.


In [6]:

# CHART 1: ESTIMATED REVENUE BY CATEGORY (Alternative to Line/Trend)
revenue_by_cat = df.groupby('root_category')['revenue_proxy'].sum().reset_index().sort_values(by='revenue_proxy', ascending=False)

fig_cat_revenue = px.bar(
    revenue_by_cat,
    x='root_category',
    y='revenue_proxy',
    title='Estimated Revenue Distribution Across Root Categories',
    labels={'root_category': 'Product Category', 'revenue_proxy': 'Estimated Revenue (Price x Reviews)'},
    template='plotly_white',
    color='revenue_proxy',
    color_continuous_scale='Viridis'
)
fig_cat_revenue.update_layout(coloraxis_showscale=False)
fig_cat_revenue.show()

# -------------------------------------------------------------
# CHART 2: TOP 10 PRODUCTS BY ESTIMATED REVENUE
# -------------------------------------------------------------
top_10_products = df.sort_values(by='revenue_proxy', ascending=False).head(10)

# Truncate long product names to 40 characters for clean presentation layout
top_10_products['short_name'] = top_10_products['product_name'].str.slice(0, 40) + "..."

fig_top_products = px.bar(
    top_10_products,
    x='revenue_proxy',
    y='short_name',
    orientation='h',
    title='Top 10 Marketplace Products by Estimated Revenue',
    labels={'revenue_proxy': 'Estimated Revenue', 'short_name': 'Product Description'},
    template='plotly_white',
    color='revenue_proxy',
    color_continuous_scale='Blues'
)
fig_top_products.update_layout(yaxis={'categoryorder':'total ascending'}, coloraxis_showscale=False)
fig_top_products.show()

# -------------------------------------------------------------
# CHART 3: MARKET CATEGORY STRUCTURE (Geographic Territory Alternative)
# -------------------------------------------------------------
fig_treemap = px.treemap(
    df,
    path=['root_category'],
    values='revenue_proxy',
    title='Market Share Share Tree Map (Category Space Dominance)',
    template='plotly_white',
    color='rating',
    color_continuous_scale='RdYlGn'
)
fig_treemap.show()


In [7]:
df.to_csv("amazon_cleaned.csv", index=False)
